# IBM Applied Data Science Capstone
## Falcon 9 landing prediction - exploratory SQL analysis

**Learner:** Djessi Jorge  
**Completed:** 4 August 2026

SQLite is used here to make every query locally reproducible. The source is the completed
101-row web-scraped launch table.

In [1]:
import sqlite3
import pandas as pd

launches = pd.read_csv("Spacex.csv")
launches = launches.rename(columns={"Landing _Outcome": "Landing_Outcome"})
for column in launches.columns:
    if pd.api.types.is_string_dtype(launches[column]):
        launches[column] = launches[column].astype(str).str.strip()

connection = sqlite3.connect(":memory:")
launches.to_sql("SPACEXTBL", connection, index=False, if_exists="replace")

def query(sql):
    return pd.read_sql_query(sql, connection)

print(f"Rows loaded into SPACEXTBL: {len(launches)}")

Rows loaded into SPACEXTBL: 101


### 1. Which launch-site labels occur in the source?

In [2]:
query('''
    SELECT DISTINCT Launch_Site
    FROM SPACEXTBL
    ORDER BY Launch_Site;
''')

,Launch_Site
0,CCAFS LC-40
1,CCAFS SLC-40
2,KSC LC-39A
3,VAFB SLC-4E


### 2. How much payload was carried for NASA Commercial Resupply Services?

In [3]:
query('''
    SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass_kg
    FROM SPACEXTBL
    WHERE Customer LIKE '%NASA (CRS)%';
''')

,Total_Payload_Mass_kg
0,48213


### 3. What was the average payload mass for the exact `F9 v1.1` booster label?

In [4]:
query('''
    SELECT ROUND(AVG(PAYLOAD_MASS__KG_), 1) AS Average_Payload_Mass_kg
    FROM SPACEXTBL
    WHERE Booster_Version = 'F9 v1.1';
''')

,Average_Payload_Mass_kg
0,2928.4


### 4. When did the first successful ground-pad landing occur?

In [5]:
query('''
    SELECT MIN(Date) AS First_Successful_Ground_Landing
    FROM SPACEXTBL
    WHERE Landing_Outcome = 'Success (ground pad)';
''')

,First_Successful_Ground_Landing
0,01-05-2017


### 5. Which boosters carried 4,000-6,000 kg and landed successfully on a drone ship?

In [6]:
query('''
    SELECT Booster_Version, PAYLOAD_MASS__KG_
    FROM SPACEXTBL
    WHERE Landing_Outcome = 'Success (drone ship)'
      AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000
    ORDER BY PAYLOAD_MASS__KG_;
''')

,Booster_Version,PAYLOAD_MASS__KG_
0,F9 FT B1026,4600
1,F9 FT B1022,4696
2,F9 FT B1031.2,5200
3,F9 FT B1021.2,5300


### 6. How many landing outcomes begin with `Success`?

In [7]:
query('''
    SELECT
        SUM(CASE WHEN Landing_Outcome LIKE 'Success%' THEN 1 ELSE 0 END)
            AS Successful_Landings,
        SUM(CASE WHEN Landing_Outcome NOT LIKE 'Success%' THEN 1 ELSE 0 END)
            AS Other_Outcomes
    FROM SPACEXTBL;
''')

,Successful_Landings,Other_Outcomes
0,61,40


### 7. Which booster records carried the maximum payload?

In [8]:
query('''
    SELECT Booster_Version, PAYLOAD_MASS__KG_
    FROM SPACEXTBL
    WHERE PAYLOAD_MASS__KG_ = (
        SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL
    )
    ORDER BY Booster_Version;
''')

,Booster_Version,PAYLOAD_MASS__KG_
0,F9 B5 B1048.4,15600
1,F9 B5 B1048.5,15600
2,F9 B5 B1049.4,15600
3,F9 B5 B1049.5,15600
4,F9 B5 B1049.7,15600
5,F9 B5 B1051.3,15600
6,F9 B5 B1051.4,15600
7,F9 B5 B1051.6,15600
8,F9 B5 B1056.4,15600
9,F9 B5 B1058.3,15600


### 8. How did landing modes rank before 20 March 2017?

In [9]:
query('''
    SELECT Landing_Outcome, COUNT(*) AS Outcome_Count
    FROM SPACEXTBL
    WHERE (
        SUBSTR(Date, 7, 4) || '-' || SUBSTR(Date, 4, 2) || '-' || SUBSTR(Date, 1, 2)
    ) BETWEEN '2010-06-04' AND '2017-03-20'
    GROUP BY Landing_Outcome
    ORDER BY Outcome_Count DESC, Landing_Outcome;
''')

,Landing_Outcome,Outcome_Count
0,No attempt,10
1,Failure (drone ship),5
2,Success (drone ship),5
3,Controlled (ocean),3
4,Success (ground pad),3
5,Failure (parachute),2
6,Uncontrolled (ocean),2
7,Precluded (drone ship),1


### Key findings

- The historical source contains four launch-site labels, including two labels for Cape Canaveral.
- NASA CRS missions carried 48,213 kg in total; exact `F9 v1.1` rows averaged 2,928.4 kg.
- Sixty-one landing outcomes begin with `Success`, while 40 are other outcomes.
- Maximum payload is 15,600 kg across 12 booster records.